<a href="https://colab.research.google.com/github/Gizeh-Gutierrez/Maestria_programacion/blob/main/Sesion10_Data_Profiling_Entregable_271752.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Gizeh Gutierrez

**Matrícula:** 271752

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [1]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

Revision de df_marketing utilizando print

In [2]:
print(df_marketing.columns)

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')


Se realiza copia y se cambian los nombres a minisculas

In [3]:
# Crear copia
df_marketing_renombrado = df_marketing.copy()
# Pasar todos los nombres a minúsculas
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()
# Revisar resultado
print(df_marketing_renombrado.columns)

Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome',
       'teenhome', 'dt_customer', 'recency', 'mntwines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')


Mejorar los nombres de la variables

In [10]:
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
'kidhome': 'kid_at_home',
'teenhome': 'teen_at_home',
'mntwines': 'mn_wines',
'mntfruits': 'mnt_fruits',
'mntmeatproducts': 'mnt_meat_products',
'mntfishproducts': 'mnt_fish_products',
'mntsweetproducts': 'mnt_sweet_products',
'mntgoldprods': 'mnt_gold_prods',
'mntgoldprods': 'mnt_gold_prods',
'numdealspurchases': 'num_deals_purchases',
'numwebpurchases': 'num_web_purchases',
'numcatalogpurchases': 'num_catalog_purchases',
'numwebvisitsmonth': 'num_web_visits_month',
'numstorepurchases': 'num_store_purchases',
'acceptedcmp3': 'accepted_cmp3',
'acceptedcmp4': 'accepted_cmp4',
'acceptedcmp5': 'accepted_cmp5',
'acceptedcmp1': 'accepted_cmp1',
'acceptedcmp2': 'accepted_cmp2',
'z_costcontact': 'z_cost_contact',


})

Verificacion de los cambios

In [11]:
print(df_marketing_renombrado.columns.tolist())

['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_at_home', 'teen_at_home', 'dt_customer', 'recency', 'amount_spent_wines', 'mnt_fruits', 'mnt_meat_products', 'mnt_fish_products', 'mnt_sweet_products', 'mnt_gold_prods', 'num_deals_purchases', 'num_web_purchases', 'num_catalog_purchases', 'num_store_purchases', 'num_web_visits_month', 'accepted_cmp3', 'accepted_cmp4', 'accepted_cmp5', 'accepted_cmp1', 'accepted_cmp2', 'complain', 'z_cost_contact', 'z_revenue', 'response']


---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [14]:
# 1 Nulos antes de convertir
nulos_antes = df_netflix['date_added'].isna().sum()
print("Nulos antes:", nulos_antes)

# 2 Convertir la columna a fecha
df_netflix['date_added'] = pd.to_datetime(
df_netflix['date_added'],
format='mixed'
)
# 3 Verificar dato
print(df_netflix.dtypes)

# 4 Nulos después de convertir
nulos_despues = df_netflix['date_added'].isna().sum()
print("Nulos después:", nulos_despues)


Nulos antes: 10
show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object
Nulos después: 10


In [16]:
print(f"Nulos antes: {nulos_antes}")
print(f"Nulos después: {nulos_despues}")
print(f"Nuevos nulo: {nulos_despues - nulos_antes}")

Nulos antes: 10
Nulos después: 10
Nuevos nulo: 0


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [17]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

se cuentan filas completamente repetidas.
se cuentan registros con un ID repetido.
se elimina las copias y devuelve el DataFrame a sus 2240 filas originales.

In [20]:
# 1 Duplicados
dup_exactos = df_marketing_dup.duplicated().sum()
print("Duplicados exactos:", dup_exactos)

# 2 Duplicados ID
dup_id = df_marketing_dup.duplicated(subset='ID').sum()
print("Duplicados ID:", dup_id)

# 3 Eliminar duplicados
df_marketing_limpio = df_marketing_dup.drop_duplicates()

# 4 Verificar
print("Filas finales:", len(df_marketing_limpio))


Duplicados exactos: 2
Duplicados ID: 2
Filas finales: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [21]:
# Valores faltantes por columna
faltantes_por_columna = df_netflix.isnull().sum()
print(faltantes_por_columna)


show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64


In [22]:
# True si la fila tiene al menos un nulo
filas_con_nulos = df_netflix.isnull().any(axis=1)

# Cantidad total de filas con al menos un nulo
total_filas_con_nulos = filas_con_nulos.sum()
print("Filas con al menos un valor faltante:", total_filas_con_nulos)

Filas con al menos un valor faltante: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [23]:
# Total de filas
total_filas = len(df_netflix)

# Nulos por columna
nulos = df_netflix.isnull().sum()

# Completitud
completitud = (1 - nulos / total_filas) * 100
print(completitud.sort_values())


director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
rating           99.910107
title           100.000000
show_id         100.000000
type            100.000000
release_year    100.000000
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64


In [26]:
print("Columna con menor completitud:", completitud.idxmin())

Columna con menor completitud: director


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [27]:
df_marketing['Marital_Status'].value_counts()


,count
Marital_Status,
Married,864
Together,580
Single,480
Divorced,232
Widow,77
Alone,3
Absurd,2
YOLO,2


Reclasificaría Alone como Single y eliminaría Absurd y YOLO, porque parecen errores que tienen pocos registros que ademas no representan estados civiles reales y pueden afectar el análisis.

In [29]:
# Crear copia
df_marketing_limpio = df_marketing.copy()

# Reclasificar Alone como Single
df_marketing_limpio['Marital_Status'] = df_marketing_limpio['Marital_Status'].replace(
{'Alone': 'Single'}
)
# Eliminar Absurd y YOLO
df_marketing_limpio = df_marketing_limpio[
~df_marketing_limpio['Marital_Status'].isin(['Absurd', 'YOLO'])
]
# Verificar resultado
df_marketing_limpio['Marital_Status'].value_counts()


,count
Marital_Status,
Married,864
Together,580
Single,483
Divorced,232
Widow,77


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [32]:
# Verificar si cada show_id cumple el patrón s + dígitos
cumple_patron = df_netflix['show_id'].str.match(r'^s\d+$')

# Porcentaje de cumplimiento
porcentaje_cumplimiento = cumple_patron.mean() * 100
print(f"Porcentaje de cumplimiento: {porcentaje_cumplimiento:.2f}%")

Porcentaje de cumplimiento: 100.00%


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [33]:

print(df_marketing[['Year_Birth']].describe())

df_marketing.sort_values('Year_Birth').head(10)


        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
239,11004,1893,2n Cycle,Single,60182.0,0,1,2014-05-17,23,8,...,4,0,0,0,0,0,0,3,11,0
339,1150,1899,PhD,Together,83532.0,0,0,2013-09-26,36,755,...,1,0,0,1,0,0,0,3,11,0
192,7829,1900,2n Cycle,Divorced,36640.0,1,0,2013-09-26,99,15,...,5,0,0,0,0,0,1,3,11,0
1950,6663,1940,PhD,Single,51141.0,0,0,2013-07-08,96,144,...,5,0,0,0,0,0,0,3,11,0
424,6932,1941,PhD,Married,93027.0,0,0,2013-04-13,77,1285,...,2,0,0,1,0,0,0,3,11,0
1923,4994,1943,Master,Single,77598.0,0,0,2013-10-01,53,1193,...,3,0,0,1,0,0,0,3,11,0
415,7106,1943,PhD,Married,75865.0,0,0,2014-03-31,73,483,...,1,0,0,0,0,0,0,3,11,0
894,8800,1943,PhD,Divorced,48948.0,0,0,2013-02-01,53,437,...,6,1,0,0,0,0,0,3,11,1
39,2968,1943,PhD,Divorced,48948.0,0,0,2013-02-01,53,437,...,6,1,0,0,0,0,0,3,11,1
1150,1453,1943,PhD,Widow,57513.0,0,0,2013-07-06,59,735,...,6,0,0,0,0,0,0,3,11,0


Al menos dos fechas de nacimiento dan como resultado  una edad mayor a 100 lo cual es improbable por lo que se deberian  revisar esos datos o en su defecto eliminarlos.

---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [34]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [38]:
# Paso 1 — ajuste de tipos

print(df_practica.dtypes)
# Identificar y corregir Income
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
print("\nTipos después de la corrección:")
print(df_practica.dtypes)


ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object

Tipos después de la corrección:
ID                       int64
Year_Birth               int64
Education            

In [39]:
# Paso 2 — duplicados

duplicados = df_practica.duplicated().sum()
print("\nDuplicados encontrados:", duplicados)

# Eliminar duplicados
df_practica = df_practica.drop_duplicates()
print("Filas después de eliminar duplicados:", len(df_practica))


Duplicados encontrados: 0
Filas después de eliminar duplicados: 15


In [40]:
# Paso 3 — valores faltantes

nulos_por_columna = df_practica.isnull().sum()
print(nulos_por_columna)

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [37]:
# Paso 4 — exploración categórica

print("\nValores únicos de Marital_Status:")
print(df_practica['Marital_Status'].unique())
print("\nFrecuencia de categorías:")
print(df_practica['Marital_Status'].value_counts())


Valores únicos de Marital_Status:
['Together' 'Single' 'Married' 'Divorced']

Frecuencia de categorías:
Marital_Status
Together    6
Married     5
Single      3
Divorced    1
Name: count, dtype: int64


**Tu reporte de profiling:**

Durante esta tare  de profiling se revisaron aspectos sobre la calidad de datos en los datasets Customer Personality Analysis y Netflix. Se estandarizaron nombres de columnas, se validaron tipos de datos y se corrigieron inconsistencias, identificando valores que se transformaron en nulos.

También se detectaron y eliminaron registros duplicados, se exploraron variables como Marital_Status, identificando categorías erroneas como Alone, Absurd y YOLO.